# Cross-Entropy and Negative Log-Likelihood

This notebook demonstrates that cross-entropy loss is equivalent to negative log-likelihood in PyTorch.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Multi-class Classification: CrossEntropyLoss = Negative Log-Likelihood

For multi-class classification with classes $C$:

$$\mathcal{L} = -\log P(y^* | x) = -\log \frac{\exp(s_{y^*})}{\sum_{c=1}^{C} \exp(s_c)}$$

where $s_c$ are the raw scores (logits) for each class.

In [2]:
# Example: 3 samples, 4 classes
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],  # sample 1
    [0.5, 2.5, 0.2, 0.1],  # sample 2
    [0.1, 0.2, 3.0, 0.5]   # sample 3
])

# True labels (class indices)
targets = torch.tensor([0, 1, 2])

print("Logits (raw scores):")
print(logits)
print("\nTrue labels:", targets.tolist())

Logits (raw scores):
tensor([[2.0000, 1.0000, 0.1000, 0.5000],
        [0.5000, 2.5000, 0.2000, 0.1000],
        [0.1000, 0.2000, 3.0000, 0.5000]])

True labels: [0, 1, 2]


### Method 1: Using CrossEntropyLoss

In [3]:
# CrossEntropyLoss combines LogSoftmax and NLLLoss
ce_loss = nn.CrossEntropyLoss()
loss_ce = ce_loss(logits, targets)

print(f"CrossEntropyLoss: {loss_ce.item():.6f}")

CrossEntropyLoss: 0.339068


### Method 2: Manual computation of negative log-likelihood

In [4]:
# Step 1: Compute softmax probabilities
probs = F.softmax(logits, dim=1)
print("Probabilities (after softmax):")
print(probs)

# Step 2: Extract probabilities for correct classes
correct_probs = probs[range(len(targets)), targets]
print("\nProbabilities for correct classes:", correct_probs.tolist())

# Step 3: Compute negative log-likelihood
nll = -torch.log(correct_probs)
print("\nNegative log-likelihoods:", nll.tolist())

# Step 4: Average across samples
loss_manual = nll.mean()
print(f"\nManual NLL (averaged): {loss_manual.item():.6f}")

Probabilities (after softmax):
tensor([[0.5745, 0.2114, 0.0859, 0.1282],
        [0.1020, 0.7540, 0.0756, 0.0684],
        [0.0459, 0.0508, 0.8348, 0.0685]])

Probabilities for correct classes: [0.5745217204093933, 0.7539703845977783, 0.8347814679145813]

Negative log-likelihoods: [0.5542173981666565, 0.2824021875858307, 0.18058530986309052]

Manual NLL (averaged): 0.339068


### Method 3: Using LogSoftmax + NLLLoss

In [5]:
# This is what CrossEntropyLoss does internally
log_probs = F.log_softmax(logits, dim=1)
nll_loss = nn.NLLLoss()
loss_nll = nll_loss(log_probs, targets)

print(f"NLLLoss (with LogSoftmax): {loss_nll.item():.6f}")

NLLLoss (with LogSoftmax): 0.339068


### Verify they are all the same

In [6]:
print(f"CrossEntropyLoss:     {loss_ce.item():.10f}")
print(f"Manual NLL:           {loss_manual.item():.10f}")
print(f"LogSoftmax + NLLLoss: {loss_nll.item():.10f}")
print(f"\nAll equal? {torch.allclose(loss_ce, loss_manual) and torch.allclose(loss_ce, loss_nll)}")

CrossEntropyLoss:     0.3390682936
Manual NLL:           0.3390682936
LogSoftmax + NLLLoss: 0.3390682936

All equal? True


## Binary Classification: BCELoss

For binary classification with labels $y \in \{0, 1\}$:

$$\mathcal{L} = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

where $\hat{y} = \sigma(s)$ is the predicted probability after sigmoid.

In [7]:
# Example: 5 samples, binary classification
logits_binary = torch.tensor([2.0, -1.0, 0.5, -2.0, 1.5])
targets_binary = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])

print("Logits (raw scores):", logits_binary.tolist())
print("True labels:", targets_binary.tolist())

Logits (raw scores): [2.0, -1.0, 0.5, -2.0, 1.5]
True labels: [1.0, 0.0, 1.0, 0.0, 1.0]


### Method 1: BCEWithLogitsLoss (more numerically stable)

In [8]:
# BCEWithLogitsLoss combines sigmoid and BCELoss for numerical stability
bce_logits_loss = nn.BCEWithLogitsLoss()
loss_bce_logits = bce_logits_loss(logits_binary, targets_binary)

print(f"BCEWithLogitsLoss: {loss_bce_logits.item():.6f}")

BCEWithLogitsLoss: 0.248522


### Method 2: Manual sigmoid + BCELoss

In [9]:
# Step 1: Apply sigmoid to get probabilities
probs_binary = torch.sigmoid(logits_binary)
print("Probabilities (after sigmoid):", probs_binary.tolist())

# Step 2: Compute BCE loss
bce_loss = nn.BCELoss()
loss_bce = bce_loss(probs_binary, targets_binary)

print(f"\nBCELoss (with sigmoid): {loss_bce.item():.6f}")

Probabilities (after sigmoid): [0.8807970285415649, 0.2689414322376251, 0.622459352016449, 0.11920291930437088, 0.8175744414329529]

BCELoss (with sigmoid): 0.248522


### Method 3: Manual computation of binary cross-entropy

In [10]:
# Manual BCE formula: -[y * log(p) + (1-y) * log(1-p)]
bce_manual = -(targets_binary * torch.log(probs_binary) + 
               (1 - targets_binary) * torch.log(1 - probs_binary))

print("Per-sample BCE:", bce_manual.tolist())
loss_bce_manual = bce_manual.mean()
print(f"\nManual BCE (averaged): {loss_bce_manual.item():.6f}")

Per-sample BCE: [0.12692806124687195, 0.31326165795326233, 0.4740769565105438, 0.12692800164222717, 0.2014133185148239]

Manual BCE (averaged): 0.248522


### Verify binary cross-entropy methods are equivalent

In [11]:
print(f"BCEWithLogitsLoss: {loss_bce_logits.item():.10f}")
print(f"BCELoss:           {loss_bce.item():.10f}")
print(f"Manual BCE:        {loss_bce_manual.item():.10f}")
print(f"\nAll equal? {torch.allclose(loss_bce_logits, loss_bce) and torch.allclose(loss_bce, loss_bce_manual)}")

BCEWithLogitsLoss: 0.2485216111
BCELoss:           0.2485215962
Manual BCE:        0.2485215962

All equal? True


## Relationship between Binary and Multi-class

Binary cross-entropy is a special case of multi-class cross-entropy when $C=2$.

In [10]:
# Binary example: 3 samples
logits_bin = torch.tensor([2.0, -1.0, 0.5])
targets_bin = torch.tensor([1.0, 0.0, 1.0])

# Equivalent multi-class formulation (2 classes)
# For class 0, logit = 0; for class 1, logit = logit_bin
logits_multi = torch.stack([torch.tensor([0, 0, 0]), logits_bin], dim=1)
targets_multi = targets_bin.long()

print("Binary logits:", logits_bin.tolist())
print("Multi-class logits (2 classes):\n", logits_multi)

# Compute losses
bce_logits = nn.BCEWithLogitsLoss()
ce = nn.CrossEntropyLoss()

loss_binary = bce_logits(logits_bin, targets_bin)
loss_multiclass = ce(logits_multi, targets_multi)

print(f"\nBinary CE Loss: {loss_binary.item():.10f}")
print(f"Multi-class CE Loss (C=2): {loss_multiclass.item():.10f}")
print(f"\nApproximately equal? {torch.allclose(loss_binary, loss_multiclass, atol=1e-6)}")

Binary logits: [2.0, -1.0, 0.5]
Multi-class logits (2 classes):
 tensor([[ 0.0000,  2.0000],
        [ 0.0000, -1.0000],
        [ 0.0000,  0.5000]])

Binary CE Loss: 0.3047555983
Multi-class CE Loss (C=2): 0.3047555387

Approximately equal? True


## Key Takeaways

1. **CrossEntropyLoss = Negative Log-Likelihood**: They are mathematically identical.

2. **CrossEntropyLoss = LogSoftmax + NLLLoss**: PyTorch's `CrossEntropyLoss` combines these for numerical stability.

3. **BCELoss**: For binary classification, use `BCEWithLogitsLoss` (combines sigmoid + BCE for stability).

4. **Binary is a special case**: Binary cross-entropy is equivalent to multi-class cross-entropy with 2 classes.

5. **Why use CrossEntropyLoss?** It's more numerically stable than computing softmax and log separately.